# StealthyIMU Full-Fledged Training Pipeline (Kaggle Environment)

This notebook provides the complete pipeline to train the **StealthyIMU** model end-to-end on the full dataset for **30 epochs** using a GPU accelerator (e.g., NVIDIA T4) on Kaggle.

It is configured to run both training phases described in the research paper:
1. **Phase 1: Teacher Model Baseline**: SpeechBrain-based transfer learning using the exact architecture from Table VII of the paper.
2. **Phase 2: Student Model Distillation**: Compressing the trained 36MB Teacher model into a lightweight 2MB Student model using Kullback-Leibler (KL) Divergence.

### Step 1: Install Dependencies
First, we install all the required Python packages on the Kaggle runtime environment.

In [ ]:
# Install dependencies in non-interactive mode
!pip install -q speechbrain hyperpyyaml sentencepiece librosa soundfile jsonlines pandas matplotlib

### Step 2: Extract Code and Setup Directories
Upload the `StealthyIMU_Kaggle_All_Phases.zip` archive into Kaggle, extract it into the working directory, and check file availability.

In [ ]:
import os
import zipfile

# Unzip the uploaded code structure if running from the zip file
zip_name = "StealthyIMU_Kaggle_All_Phases.zip"
if os.path.exists(zip_name):
    print(f"Found code archive: {zip_name}. Extracting...")
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Code archive extracted successfully!")
else:
    print("Code directory already extracted or zip not present in local working dir. Listing current files:")
    !ls -la

# Ensure hparams and pretrain directories exist
os.makedirs("results", exist_ok=True)
assert os.path.exists("hparams"), "Missing hparams folder!"
assert os.path.exists("pretrain"), "Missing pretrain/ folder with vocabulary model!"
print("Project directory structure validated successfully.")

### Step 3: Procure the Dataset
The dataset is large and should be fetched from Google Drive. Uncomment the helper code below and input your Google Drive File ID to download the precomputed dataset directly into Kaggle.

In [ ]:
# Optional: Install gdown and download dataset from Google Drive
# !pip install -q gdown
# import gdown
# 
# # Replace with your actual Google Drive File ID:
# file_id = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"
# output_zip = "StealthyIMU_dataset.zip"
# 
# print("Downloading dataset from Google Drive...")
# gdown.download(f'https://drive.google.com/uc?id={file_id}', output_zip, quiet=False)
# 
# # Unzip dataset into working directory
# if os.path.exists(output_zip):
#     print("Extracting StealthyIMU dataset...")
#     with zipfile.ZipFile(output_zip, 'r') as zip_ref:
#         zip_ref.extractall(".")
#     print("Dataset successfully extracted!")
# else:
#     print("Please upload the dataset or configure the data path manually.")

### Step 4: Verify GPU and CUDA Accelerator
Ensure PyTorch detects the CUDA device for GPU acceleration.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA accelerator available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("WARNING: GPU is not available! Training will run on CPU and be extremely slow.")
    device = "cpu"

## Phase 1: Teacher Model Training (30 Epochs)
We launch the full baseline SpeechBrain training configured matching Table VII of the research paper (1 CNN Block, 4 LSTM layers). The training runs on GPU for the full-fledged **30 epochs**.

In [ ]:
# Run the training script for the full 30 epochs on GPU
!python train.py hparams/paper_exact.yaml --number_of_epochs 30 --device {device}

## Phase 2: Student Model Distillation (30 Epochs)
Once the Phase 1 Teacher model is trained and its checkpoint is saved at `results/slu_baseline_paper/1235/save/model.ckpt`, we run the Knowledge Distillation wrapper. The Student model will train for **30 epochs** on the GPU, matching the paper's target specs.

In [ ]:
# Verify that teacher model weights exist before launching KD
teacher_ckpt = "results/slu_baseline_paper/1235/save/model.ckpt"
if os.path.exists(teacher_ckpt):
    print("Validated Phase 1 Teacher checkpoint. Starting distillation...")
else:
    print("WARNING: Teacher model checkpoint not found at the expected path.")
    print("Please verify the output directory of Phase 1 before running this cell.")

# Start student model distillation training
!python run_phase2_kd.py hparams/phase2_kd.yaml --number_of_epochs 30 --device {device}

## Step 5: Metrics & Curve Visualization
We parse the training log text file to plot training and validation losses, along with the validation Word Error Rate (WER) across epochs.

In [ ]:
import os
import matplotlib.pyplot as plt

def parse_train_log(log_path):
    epochs, train_losses, valid_losses, valid_wer = [], [], [], []
    if not os.path.exists(log_path):
        print(f"Log file not found: {log_path}")
        return None
    
    with open(log_path, "r") as f:
        for line in f:
            if "epoch:" in line:
                # Parse basic metadata: epoch, lr, loss
                parts = line.strip().split(" - ")
                # Epoch info
                e_info = parts[0].split(", ")
                epoch = int(e_info[0].split(": ")[1])
                epochs.append(epoch)
                
                # Loss info
                l_info = parts[1].split(", ")
                t_loss = float(l_info[0].split(": ")[1])
                v_loss = float(l_info[1].split(": ")[1])
                train_losses.append(t_loss)
                valid_losses.append(v_loss)
                
                # Parse WER and CER
                for info_part in l_info:
                    if "WER:" in info_part:
                        try:
                            wer_val = float(info_part.split("WER: ")[1].replace("%", ""))
                            valid_wer.append(wer_val)
                        except:
                            pass
    return epochs, train_losses, valid_losses, valid_wer

# Plot Phase 1 curves
p1_stats = parse_train_log("results/slu_baseline_paper/1235/train_log.txt")
if p1_stats:
    epochs, t_losses, v_losses, v_wers = p1_stats
    plt.figure(figsize=(14, 5))
    
    # Loss plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs, t_losses, label="Train Loss", marker='o')
    plt.plot(epochs, v_losses, label="Validation Loss", marker='s')
    plt.title("Phase 1: Loss vs Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    
    # WER plot
    if len(v_wers) == len(epochs):
        plt.subplot(1, 2, 2)
        plt.plot(epochs, v_wers, label="Val WER (%)", color="orange", marker='^')
        plt.title("Phase 1: Validation WER (%) vs Epochs")
        plt.xlabel("Epoch")
        plt.ylabel("WER (%)")
        plt.grid(True)
        plt.legend()
        
    plt.tight_layout()
    plt.show()
else:
    print("No metrics plotted. Complete training first.")